# 06 (LARGE) - Conflict-Gated Slot-Attributed PLM (CG-SAP Large) — bản Google Colab + Ensemble nhiều seed

Bản chạy trên **Google Colab** của mô hình CG-SAP Large (Ensemble đa hạt giống):
- **Tự động Mount Google Drive**: Checkpoints và kết quả lưu bền vững vào Google Drive (không lo mất khi ngắt kết nối session).
- **Dữ liệu linh hoạt**: Tự động nhận diện file dữ liệu trên Google Drive (`/content/drive/MyDrive/...`), thư mục cục bộ (`./dataset_raw`), hoặc tự động tải về qua `kagglehub` (`thaonngyn/data-fc`).
- **Chọn tập dữ liệu**: Đổi cờ `DATASET_NAME = 'vifactcheck' | 'viwikifc'` tại Cell cấu hình.
- **Ensemble 5 seed (`ENSEMBLE_SEEDS = [1, 10, 100, 1000, 10000]`)**: Dự đoán trung bình softmax, xuất đầy đủ thống kê Mean ± Std và Bảng kết quả SOTA.
- **Cải tiến tối ưu (Ablation components)**:
  1. **Fine-tune trung gian NLI tiếng Việt** (XNLI-vi) để chuyển giao tri thức suy luận logic trước khi vào bài toán Fact-Checking.
  2. **Đánh dấu khác biệt trong input**: từ của claim không có trong evidence được bao bởi `<diff> … </diff>`.
  3. **Pooling riêng claim / evidence**: verdict head nhận `[cls; c; e; |c−e|; c⊙e]` thay vì chỉ `[CLS]`.
  4. **Cổng điều biến mâu thuẫn (Conflict Gate)** kết hợp **Hàm mất mát nhất quán logic (Consistency Loss)**.
  5. **Label smoothing 0.1**, **R-Drop**, **dropout head 0.1**.
  6. **Differential Learning Rate & Head Warmup**: Phân tách LR (Backbone 5e-6, Heads 1e-4) giúp 100% các seed hội tụ ổn định.

**Thiết lập Google Colab trước khi chạy:**
1. *Runtime → Change runtime type*: Chọn Hardware accelerator là **GPU (T4, V100 hoặc A100)**.
2. Để lưu kết quả và checkpoint lâu dài, cho phép kết nối Google Drive khi được hỏi.

## 1. Môi trường & Thư viện (Google Colab)
Tự động mount Google Drive, kiểm tra GPU và cài đặt các thư viện cần thiết.

In [ ]:
import os, sys, re, json, time, unicodedata, shutil, random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any
from datetime import datetime, timezone

# 1. Tự động Mount Google Drive nếu chạy trên Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
    print('✅ Đã mount Google Drive thành công.')
except Exception:
    ON_COLAB = False
    print('💻 Chạy trên môi trường Local / Non-Colab.')

# 2. Cài đặt các thư viện cần thiết trên Colab
!pip install -q transformers datasets scikit-learn accelerate "kagglehub[pandas-datasets]"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt

ON_KAGGLE = os.path.isdir('/kaggle/working')
print(f'Môi trường - Colab: {ON_COLAB} | Kaggle: {ON_KAGGLE} | PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| Số GPU:', torch.cuda.device_count())

## 2. Cấu hình thực nghiệm tối ưu cho Large (Knobs)

In [ ]:
# ===================== CẤU HÌNH TỐI ƯU CHO LARGE =====================
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ===================== CHỌN TẬP DỮ LIỆU =====================
DATASET_NAME = 'viwikifc'      # 'vifactcheck' | 'viwikifc'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using Device: {DEVICE} | Dataset: {DATASET_NAME}')

# ===================== DỮ LIỆU TRÊN KAGGLE =====================
KAGGLE_DATASET = 'thaonngyn/data-fc'
# Tên file của từng split trong Kaggle dataset. Nếu file nằm trong thư mục con, cell dữ liệu sẽ tự tìm
# đường dẫn đầy đủ; chỉ cần sửa ở đây khi tên file khác.
DATASET_FILES = {
    'vifactcheck': {
        'train': 'train-vifactcheck.csv',
        'dev': 'dev-vifactcheck.csv',
        'test': 'test-vifactcheck.csv',
    },
    'viwikifc': {
        'train': 'train-factcheckwiki.csv',
        'dev': 'dev-factcheckwiki.csv',
        'test': 'test-factcheckwiki.csv',
    },
}

# ===================== CẢI TIẾN (bật/tắt để ablation) =====================
# 1. Fine-tune trung gian trên NLI tiếng Việt. Chạy 1 lần, lưu backbone vào WORKING_DIR, mọi seed dùng chung.
USE_NLI_INTERMEDIATE = True
NLI_SOURCE = 'xnli'        # 'xnli' = Hugging Face facebook/xnli (config 'vi') | 'csv' = file tự upload
NLI_CSV_PATH = None        # khi NLI_SOURCE='csv': CSV có cột premise, hypothesis, label (entailment/neutral/contradiction)
NLI_TRAIN_SAMPLES = 50000  # số cặp NLI dùng để train (XNLI-vi có ~392k); ~30-40 phút trên T4
NLI_EPOCHS = 2             # Số epoch huấn luyện NLI (mặc định thử nghiệm: 2 epochs; có thể đổi 1, 2, 3 tùy ý)
FORCE_RETRAIN_NLI = False  # True: Bắt buộc train lại NLI; False: Dùng lại checkpoint NLI nếu đã có sẵn
NLI_LR = 1e-5
NLI_MAX_LEN = 128
NLI_BATCH_SIZE = 16
NLI_GRAD_ACCUM = 2
NLI_SEED = 42
# 2. Đánh dấu từ của claim không có trong evidence bằng <diff> ... </diff>
USE_DIFF_MARKERS = True
# 3. Pooling riêng claim / evidence: [cls; c; e; |c-e|; c*e]
USE_PAIR_POOLING = True
# 4. Regularization
LABEL_SMOOTHING = 0.1      # 0 = tắt
R_DROP_ALPHA = 1.0         # trọng số KL giữa 2 lượt forward; 0 = tắt R-Drop

RUN_TAG = '_'.join(tag for tag, on in [
    ('nli', USE_NLI_INTERMEDIATE),
    ('diff', USE_DIFF_MARKERS),
    ('pair', USE_PAIR_POOLING),
    (f'ls{LABEL_SMOOTHING:g}', LABEL_SMOOTHING > 0),
    (f'rdrop{R_DROP_ALPHA:g}', R_DROP_ALPHA > 0),
] if on) or 'base'

# ===================== OUTPUT TRÊN COLAB / DRIVE =====================
if ON_COLAB and os.path.exists('/content/drive/MyDrive'):
    # Lưu trực tiếp vào Google Drive để không bị mất khi hết phiên Colab
    BASE_DIR = '/content/drive/MyDrive/NCKH'
elif ON_KAGGLE:
    BASE_DIR = '/kaggle/working'
else:
    BASE_DIR = os.path.abspath('./colab_working')

WORKING_DIR = os.path.join(BASE_DIR, 'main_runs')
OUT_DIR = os.path.join(WORKING_DIR, f'cgsap_large_{DATASET_NAME}_{RUN_TAG}')
os.makedirs(OUT_DIR, exist_ok=True)
print('WORKING_DIR:', WORKING_DIR)
print('OUT_DIR    :', OUT_DIR)

# ===================== ENSEMBLE NHIỀU SEED =====================
# Mỗi seed train một model độc lập; kết quả cuối = trung bình softmax của các seed.
# Mỗi seed lưu checkpoint + xác suất dev/test riêng trong OUT_DIR, nên chạy lại cell huấn luyện
# trong cùng session sẽ bỏ qua seed đã xong.
ENSEMBLE_SEEDS = [1,10, 100, 1000,10000]
# Seed đã train ở lần chạy trước: {seed: đường dẫn checkpoint}. Seed có ở đây chỉ chạy dự đoán.
# Trên Colab: trỏ đường dẫn tới file trên Google Drive, ví dụ
# {100: '/kaggle/input/cgsap-checkpoints/best_multitask_xlmr_large_seed100.pt'}
# Lưu ý: checkpoint của cấu hình cũ (không diff marker / không pair pooling) không nạp được vào kiến trúc mới.
EXISTING_CHECKPOINTS = {}
# Mỗi checkpoint XLM-R Large ~2.2 GB; lưu ý dung lượng Google Drive.
# False = xóa checkpoint sau khi đã lưu xác suất dự đoán (ensemble chỉ cần file .npz).
KEEP_CHECKPOINTS = True

# --- HYPERPARAMETERS CHO XLM-ROBERTA-LARGE ---
MODEL_KEY = 'xlm-roberta-large'

MAX_LEN = 256
# R-Drop nhân đôi batch trong mỗi lượt forward, nên giảm batch để vừa 16GB; effective batch giữ 16.
BATCH_SIZE = 4 if R_DROP_ALPHA > 0 else 8
GRAD_ACCUM = 16 // BATCH_SIZE
MAX_GRAD_NORM = 1.0
# 🌟 CƠ CHẾ ỔN ĐỊNH HÓA GRADIENT (HEAD WARMUP + DIFFERENTIAL LR)
LR_BACKBONE = 5e-6         # Tốc độ học chuẩn, cẩn trọng cho pre-trained backbone
LR_HEADS = 1e-4            # Tốc độ học nhanh hơn cho các Head & Gate mới khởi tạo ngẫu nhiên
HEAD_WARMUP_EPOCHS = 1     # Epoch 1: Khóa backbone, chỉ train Heads
NUM_EPOCHS = 8
DROPOUT = 0.1              # giảm từ 0.3
LAMBDA_CONFLICT = 0.35      # Trọng số loss đa nhiệm Head 2
LAMBDA_CONSISTENCY = 0.20   # Trọng số mất mát nhất quán logic

# Đặt None để chạy full toàn bộ tập dữ liệu (đặt số nhỏ để chạy thử nhanh)
TRAIN_LIMIT = None
DEV_LIMIT = None
TEST_LIMIT = None
# ===============================================================

## 3. Data Loader (Google Drive / Kagglehub / Local) & Gán nhãn Slot Mâu thuẫn tự động
Hỗ trợ tự động nhận diện dữ liệu từ Google Drive, thư mục local, hoặc tự động tải về từ `kagglehub`.

In [ ]:
LABEL2ID = {'Supported': 0, 'Refuted': 1, 'NEI': 2}
ID2LABEL = {0: 'Supported', 1: 'Refuted', 2: 'NEI'}

CONFLICT_MAP = {
    'NONE': 0,        # Dành cho Supported hoặc NEI
    'TEMPORAL': 1,    # Sai ngày/tháng/năm
    'NUMERIC': 2,     # Sai số lượng/đơn vị
    'ENTITY': 3,      # Sai nhân vật/thực thể/địa danh
    'NEGATION': 4,    # Sai phủ định
}
ID2CONFLICT = {v: k for k, v in CONFLICT_MAP.items()}

def extract_years(text: str) -> List[int]:
    return sorted(set(int(y) for y in re.findall(r'\b((?:16|17|18|19|20)\d{2})\b', str(text))))

def extract_numbers(text: str) -> List[float]:
    nums = []
    for raw in re.findall(r'(?<!\w)(\d+(?:[.,]\d+)*)(?!\w)', str(text)):
        try:
            n = raw.replace(' ', '').replace('.', '').replace(',', '.')
            val = float(n)
            if val < 1600 or val > 2100:
                nums.append(val)
        except ValueError:
            pass
    return sorted(set(nums))

def detect_conflict_type(claim: str, evidence: str, label_str: str) -> str:
    if label_str != 'Refuted':
        return 'NONE'

    c_years = extract_years(claim)
    e_years = extract_years(evidence)
    if c_years and e_years and (set(c_years) != set(e_years)):
        return 'TEMPORAL'

    c_nums = extract_numbers(claim)
    e_nums = extract_numbers(evidence)
    if c_nums and e_nums and (set(c_nums) != set(e_nums)):
        return 'NUMERIC'

    negations = {'không', 'chưa', 'chẳng', 'không_thể', 'chưa_từng', 'phủ_nhận'}
    c_has_neg = any(w in claim.lower() for w in negations)
    e_has_neg = any(w in evidence.lower() for w in negations)
    if c_has_neg != e_has_neg:
        return 'NEGATION'

    return 'ENTITY'

@dataclass
class MultiTaskSample:
    id: str
    claim: str
    evidence: str
    label_verdict: int
    label_conflict: int
    conflict_name: str


def find_dataset_file(fname: str) -> str:
    # 1. Quét tìm trong Google Drive, /content hoặc thư mục local
    candidate_dirs = [
        '/content/drive/MyDrive/NCKH/data',
        '/content/drive/MyDrive/data',
        '/content/drive/MyDrive/fact-checking-project/data',
        '/content/drive/MyDrive/fact-checking-project/dataset_raw',
        '/content/data',
        '/content',
        './dataset_raw',
        '../dataset_raw',
        '../../dataset_raw',
        './data',
        '/kaggle/input/data-fc',
    ]
    for d in candidate_dirs:
        p = Path(d)
        if p.exists():
            matches = list(p.rglob(fname))
            if matches:
                print(f"🔍 Tìm thấy '{fname}' tại: {matches[0]}")
                return str(matches[0])

    # 2. Tải tự động qua kagglehub nếu chưa có trên Drive / local
    try:
        import kagglehub
        print(f"📥 Đang tải dataset qua kagglehub ('{KAGGLE_DATASET}')...")
        local_root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        matches = list(local_root.rglob(fname))
        if matches:
            print(f"🔍 Tìm thấy '{fname}' trong kagglehub cache: {matches[0]}")
            return str(matches[0])
    except Exception as e:
        print(f"⚠️ Không tải được qua kagglehub: {e}")

    raise FileNotFoundError(
        f"Không tìm thấy '{fname}' trong Google Drive, local hoặc qua kagglehub ('{KAGGLE_DATASET}')."
    )

def load_split_df(dataset: str, split: str) -> pd.DataFrame:
    file_path = find_dataset_file(DATASET_FILES[dataset][split])
    print(f"📖 Đang đọc dữ liệu từ: {file_path}")
    return pd.read_csv(file_path)


def load_multitask_data(dataset: str, split: str, limit: Optional[int] = None) -> List[MultiTaskSample]:
    df = load_split_df(dataset, split)
    if limit:
        df = df.head(limit)

    samples = []
    for idx, r in df.iterrows():
        if dataset == 'vifactcheck':
            sid = str(r.get('annotation_id') or idx)
            c = str(r.get('Statement') or '').strip()
            e = str(r.get('Evidence') or '').strip()
            raw_l = r.get('labels')
            l_map = {0: 'Supported', 1: 'Refuted', 2: 'NEI', '0': 'Supported', '1': 'Refuted', '2': 'NEI'}
        else:
            sid = str(r.get('pairID') or idx)
            c = str(r.get('claim') or '').strip()
            e = str(r.get('evidence') or '').strip()
            raw_l = r.get('gold_label')
            l_map = {'Supports': 'Supported', 'Refutes': 'Refuted', 'Not_Enough_Information': 'NEI'}

        l_str = l_map.get(raw_l)
        if not l_str or not c:
            continue

        c_type = detect_conflict_type(c, e, l_str)
        samples.append(MultiTaskSample(
            id=sid, claim=c, evidence=e,
            label_verdict=LABEL2ID[l_str],
            label_conflict=CONFLICT_MAP[c_type],
            conflict_name=c_type,
        ))

    print(f'Loaded {len(samples)} samples from {dataset} [{split}]')
    return samples

train_samples = load_multitask_data(DATASET_NAME, 'train', limit=TRAIN_LIMIT)
dev_samples = load_multitask_data(DATASET_NAME, 'dev', limit=DEV_LIMIT)
test_samples = load_multitask_data(DATASET_NAME, 'test', limit=TEST_LIMIT)

## 4. PyTorch Dataset & Tokenizer


In [ ]:
from transformers import AutoTokenizer

print(f'Loading Tokenizer for {MODEL_KEY}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_KEY)

# ===================== ĐÁNH DẤU KHÁC BIỆT CLAIM ↔ EVIDENCE =====================
# Từ của claim không xuất hiện trong evidence được bao bởi <diff> ... </diff>, để model thấy ngay chỗ bị sửa
# (đổi thì, đổi lượng từ, thêm/bớt chi tiết). Không dùng nhãn nên áp dụng giống nhau lúc train và inference.
DIFF_OPEN, DIFF_CLOSE = '<diff>', '</diff>'
if USE_DIFF_MARKERS:
    tokenizer.add_special_tokens({'additional_special_tokens': [DIFF_OPEN, DIFF_CLOSE]})

_EDGE_PUNCT = re.compile(r'^\W+|\W+$')

def _norm_word(word: str) -> str:
    return _EDGE_PUNCT.sub('', word.lower())

def mark_claim_diff(claim: str, evidence: str) -> str:
    evidence_words = {_norm_word(w) for w in evidence.split()}
    out, span_open = [], False
    for word in claim.split():
        norm = _norm_word(word)
        novel = bool(norm) and norm not in evidence_words
        if novel and not span_open:
            out.append(DIFF_OPEN)
            span_open = True
        elif not novel and span_open:
            out.append(DIFF_CLOSE)
            span_open = False
        out.append(word)
    if span_open:
        out.append(DIFF_CLOSE)
    return ' '.join(out)

def resize_with_mean_init(model, vocab_size: int):
    # Token mới (<diff>, </diff>) khởi tạo bằng trung bình embedding sẵn có thay vì ngẫu nhiên.
    old_size = model.get_input_embeddings().weight.shape[0]
    if old_size == vocab_size:
        return
    model.resize_token_embeddings(vocab_size)
    with torch.no_grad():
        emb = model.get_input_embeddings().weight
        emb[old_size:] = emb[:old_size].mean(dim=0, keepdim=True)

def encode_pair(first: str, second: str, max_len: int) -> Dict[str, torch.Tensor]:
    # first = claim / hypothesis, second = evidence / premise. Trả thêm mask của từng đoạn cho pair pooling.
    if USE_DIFF_MARKERS:
        first = mark_claim_diff(first, second)
    enc = tokenizer(first, second, truncation=True, padding='max_length', max_length=max_len)
    seq_ids = enc.sequence_ids()
    return {
        'input_ids': torch.tensor(enc['input_ids'], dtype=torch.long),
        'attention_mask': torch.tensor(enc['attention_mask'], dtype=torch.long),
        'claim_mask': torch.tensor([1 if s == 0 else 0 for s in seq_ids], dtype=torch.long),
        'evidence_mask': torch.tensor([1 if s == 1 else 0 for s in seq_ids], dtype=torch.long),
    }

MODEL_INPUT_KEYS = ('input_ids', 'attention_mask', 'claim_mask', 'evidence_mask')

def model_inputs(batch) -> Dict[str, torch.Tensor]:
    return {k: batch[k].to(DEVICE) for k in MODEL_INPUT_KEYS}

class MultiTaskFactDataset(Dataset):
    def __init__(self, samples: List[MultiTaskSample], max_len: int = 256):
        self.samples = samples
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        item = encode_pair(s.claim, s.evidence, self.max_len)
        item['label_verdict'] = torch.tensor(s.label_verdict, dtype=torch.long)
        item['label_conflict'] = torch.tensor(s.label_conflict, dtype=torch.long)
        return item

train_loader = DataLoader(MultiTaskFactDataset(train_samples, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(MultiTaskFactDataset(dev_samples, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(MultiTaskFactDataset(test_samples, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
print(f'Train batches: {len(train_loader)} (effective batch = {BATCH_SIZE * GRAD_ACCUM})')
if USE_DIFF_MARKERS:
    print('Ví dụ claim sau khi đánh dấu:', mark_claim_diff(train_samples[0].claim, train_samples[0].evidence)[:300])


## 4b. Fine-tune trung gian trên NLI tiếng Việt

Fine-tune `xlm-roberta-large` trên cặp (hypothesis, premise) của XNLI-vi (hoặc CSV ViNLI tự upload) trước,
rồi dùng backbone đó cho fact-checking. Nhãn NLI được map: entailment → Supported, contradiction → Refuted,
neutral → NEI. Cùng tokenizer và cùng cách đánh dấu `<diff>` như tập fact-checking.
Chạy một lần; nếu thư mục backbone đã có thì bỏ qua. Mọi seed trong ensemble dùng chung backbone này.

In [ ]:
import gc
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

NLI_LABEL_TO_VERDICT = {
    'entailment': LABEL2ID['Supported'],
    'contradiction': LABEL2ID['Refuted'],
    'neutral': LABEL2ID['NEI'],
}
XNLI_ID_TO_NAME = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
NLI_VAL_SAMPLES = 2000
NLI_DIR = os.path.join(
    WORKING_DIR,
    f"xlmr_large_nli_{NLI_SOURCE}_{NLI_TRAIN_SAMPLES}_ep{NLI_EPOCHS}" + ('_diff' if USE_DIFF_MARKERS else '')
)


def load_nli_pairs():
    # Trả về (train_rows, val_rows), mỗi row = (hypothesis, premise, verdict_label_id).
    if NLI_SOURCE == 'xnli':
        from datasets import load_dataset
        train = load_dataset('facebook/xnli', 'vi', split='train').shuffle(seed=NLI_SEED)
        val = load_dataset('facebook/xnli', 'vi', split='validation').shuffle(seed=NLI_SEED)
        train = train.select(range(min(NLI_TRAIN_SAMPLES, len(train))))
        val = val.select(range(min(NLI_VAL_SAMPLES, len(val))))
        to_rows = lambda ds: [
            (r['hypothesis'], r['premise'], NLI_LABEL_TO_VERDICT[XNLI_ID_TO_NAME[int(r['label'])]]) for r in ds
        ]
        return to_rows(train), to_rows(val)

    if NLI_SOURCE == 'csv':
        df = pd.read_csv(NLI_CSV_PATH)
        df['label'] = df['label'].astype(str).str.strip().str.lower()
        df = df[df['label'].isin(NLI_LABEL_TO_VERDICT)].sample(frac=1.0, random_state=NLI_SEED)
        rows = [(str(r.hypothesis), str(r.premise), NLI_LABEL_TO_VERDICT[r.label]) for r in df.itertuples()]
        val_rows, train_rows = rows[:NLI_VAL_SAMPLES], rows[NLI_VAL_SAMPLES:]
        return train_rows[:NLI_TRAIN_SAMPLES], val_rows

    raise ValueError(f"NLI_SOURCE không hợp lệ: {NLI_SOURCE!r} (chọn 'xnli' hoặc 'csv')")


class NLIDataset(Dataset):
    def __init__(self, rows, max_len: int):
        self.rows = rows
        self.max_len = max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        hypothesis, premise, label = self.rows[idx]
        item = encode_pair(hypothesis, premise, self.max_len)
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item


def nli_accuracy(model, loader) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch['input_ids'].to(DEVICE),
                           attention_mask=batch['attention_mask'].to(DEVICE)).logits
            correct += (logits.argmax(dim=-1).cpu() == batch['labels']).sum().item()
            total += batch['labels'].numel()
    return correct / max(total, 1)


def run_nli_intermediate():
    train_rows, val_rows = load_nli_pairs()
    print(f'NLI {NLI_SOURCE}: train {len(train_rows)} | val {len(val_rows)} -> {NLI_DIR}')
    set_seed(NLI_SEED)
    nli_model = AutoModelForSequenceClassification.from_pretrained(MODEL_KEY, num_labels=3)
    resize_with_mean_init(nli_model, len(tokenizer))
    nli_model.to(DEVICE)

    loader = DataLoader(NLIDataset(train_rows, NLI_MAX_LEN), batch_size=NLI_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(NLIDataset(val_rows, NLI_MAX_LEN), batch_size=NLI_BATCH_SIZE * 2, shuffle=False)
    optimizer = torch.optim.AdamW(nli_model.parameters(), lr=NLI_LR, weight_decay=0.01, eps=1e-8)
    total_steps = max(1, len(loader) * NLI_EPOCHS // NLI_GRAD_ACCUM)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.06), total_steps)
    scaler = torch.cuda.amp.GradScaler()

    best_val_acc = 0.0
    for epoch in range(NLI_EPOCHS):
        nli_model.train()
        t0, running = time.time(), 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(loader):
            with torch.cuda.amp.autocast():
                out = nli_model(input_ids=batch['input_ids'].to(DEVICE),
                                attention_mask=batch['attention_mask'].to(DEVICE),
                                labels=batch['labels'].to(DEVICE))
                loss = out.loss / NLI_GRAD_ACCUM
            scaler.scale(loss).backward()
            running += out.loss.item()
            if (step + 1) % NLI_GRAD_ACCUM == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(nli_model.parameters(), MAX_GRAD_NORM)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad()
            if (step + 1) % 500 == 0:
                print(f'  [NLI] epoch {epoch+1}/{NLI_EPOCHS} step {step+1}/{len(loader)} | loss {running/(step+1):.4f} | '
                      f'{time.time()-t0:.0f}s')
        val_acc = nli_accuracy(nli_model, val_loader)
        print(f'⭐ [NLI] Epoch {epoch+1}/{NLI_EPOCHS} | train loss {running/len(loader):.4f} | '
              f'val acc {val_acc*100:.2f}% | {time.time()-t0:.0f}s')

        # Lưu checkpoint tốt nhất theo val_accuracy qua các epoch
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs(NLI_DIR, exist_ok=True)
            nli_model.base_model.save_pretrained(NLI_DIR)
            tokenizer.save_pretrained(NLI_DIR)
            with open(os.path.join(NLI_DIR, 'nli_intermediate_info.json'), 'w', encoding='utf-8') as f:
                json.dump({
                    'source': NLI_SOURCE,
                    'train_samples': len(train_rows),
                    'total_epochs': NLI_EPOCHS,
                    'best_epoch': epoch + 1,
                    'lr': NLI_LR,
                    'max_len': NLI_MAX_LEN,
                    'diff_markers': USE_DIFF_MARKERS,
                    'best_val_accuracy': round(best_val_acc * 100, 2),
                    'saved_at': datetime.now(timezone.utc).isoformat()
                }, f, indent=2)
            print(f'  -> Đã lưu Best NLI checkpoint tại Epoch {epoch+1} (Val Acc: {best_val_acc*100:.2f}%)')
    del nli_model, optimizer
    gc.collect()
    torch.cuda.empty_cache()


BACKBONE_PATH = MODEL_KEY
if USE_NLI_INTERMEDIATE:
    should_retrain = globals().get('FORCE_RETRAIN_NLI', False) or not os.path.exists(os.path.join(NLI_DIR, 'config.json'))
    if not should_retrain:
        print(f'Đã có backbone NLI tại {NLI_DIR} — bỏ qua bước fine-tune trung gian (đặt FORCE_RETRAIN_NLI=True nếu muốn chạy lại).')
    else:
        print(f'Bắt đầu huấn luyện NLI intermediate ({NLI_EPOCHS} epochs) -> lưu vào: {NLI_DIR}')
        run_nli_intermediate()
    BACKBONE_PATH = NLI_DIR
print('BACKBONE_PATH:', BACKBONE_PATH)

## 5. Kiến trúc Conflict-Gated Slot-Attributed PLM (CG-SAP Large)
Tích hợp **Conflict-Gated Fusion Layer**: Nhánh slot mâu thuẫn trực tiếp điều phối vector đại diện trước khi đưa vào Verdict Head.


In [ ]:
from transformers import AutoModel

class ConflictGatedFactClassifier(nn.Module):
    def __init__(self, model_name: str, num_verdict_labels: int = 3, num_conflict_labels: int = 5,
                 dropout: float = 0.1, pair_pooling: bool = True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.pair_pooling = pair_pooling

        # --- NHÁNH 0: PAIR POOLING [cls; c; e; |c-e|; c*e] -> hidden_size ---
        if pair_pooling:
            self.pair_proj = nn.Sequential(
                nn.Linear(5 * hidden_size, hidden_size),
                nn.GELU(),
                nn.Dropout(dropout)
            )

        # --- NHÁNH 1: TRÍCH XUẤT ĐẶC TRƯNG SLOT MÂU THUẪN ---
        self.conflict_proj = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.conflict_head = nn.Linear(hidden_size // 2, num_conflict_labels)

        # --- NHÁNH 2: CỔNG ĐIỀU BIẾN MÂU THUẪN (CONFLICT GATE) ---
        # Gate g nằm trong khoảng (0, 1), quyết định mức độ can thiệp của mâu thuẫn vào phán quyết
        self.conflict_gate = nn.Sequential(
            nn.Linear(hidden_size // 2, hidden_size),
            nn.Sigmoid()
        )
        self.conflict_shift_proj = nn.Linear(hidden_size // 2, hidden_size)

        # --- NHÁNH 3: PHÁN QUYẾT CHÂN LÝ (VERDICT HEAD NHẬN FUSED REPRESENTATION) ---
        self.verdict_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_verdict_labels)
        )

    @staticmethod
    def _masked_mean(hidden, mask):
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1.0)

    def forward(self, input_ids, attention_mask, claim_mask=None, evidence_mask=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        hidden = outputs.last_hidden_state
        cls_rep = hidden[:, 0, :]

        if self.pair_pooling:
            claim_rep = self._masked_mean(hidden, claim_mask)
            evidence_rep = self._masked_mean(hidden, evidence_mask)
            pair = torch.cat([cls_rep, claim_rep, evidence_rep,
                              (claim_rep - evidence_rep).abs(), claim_rep * evidence_rep], dim=-1)
            rep = self.pair_proj(self.dropout(pair))
        else:
            rep = self.dropout(cls_rep)

        # 1. Trích xuất đặc trưng mâu thuẫn slot vi mô
        conflict_feat = self.conflict_proj(rep)
        logits_conflict = self.conflict_head(conflict_feat)

        # 2. Tính toán Cổng Mâu thuẫn (Conflict Gate)
        gate = self.conflict_gate(conflict_feat)                  # (B, hidden_size)
        conflict_shift = self.conflict_shift_proj(conflict_feat)  # (B, hidden_size)

        # 3. Hợp nhất: Nếu phát hiện mâu thuẫn, gate kích hoạt để bẻ lái biểu diễn ngữ cảnh
        fused_rep = rep + gate * conflict_shift
        logits_verdict = self.verdict_head(fused_rep)

        return logits_verdict, logits_conflict, gate

def build_model() -> ConflictGatedFactClassifier:
    model = ConflictGatedFactClassifier(
        BACKBONE_PATH, num_verdict_labels=3, num_conflict_labels=5,
        dropout=DROPOUT, pair_pooling=USE_PAIR_POOLING
    )
    resize_with_mean_init(model.backbone, len(tokenizer))
    return model.to(DEVICE)

print(f'Kiến trúc CG-SAP-FC sẵn sàng | backbone: {BACKBONE_PATH} | pair pooling: {USE_PAIR_POOLING} | '
      f'diff markers: {USE_DIFF_MARKERS}')


## 6. Huấn luyện Mô hình Đa nhiệm với Head Warmup & Differential LR
Áp dụng cơ chế ổn định hóa tối ưu hóa chuẩn mực (ICLR 2021):
1. **Head Warmup (Epoch 1)**: Đóng băng backbone, chỉ tối ưu Head & Gate để tránh gradient ngẫu nhiên phá vỡ biểu diễn pre-trained.
2. **Differential Learning Rate (Epoch 2-8)**: Phân tách LR (Backbone: 5e-6, Heads: 1e-4) giúp mô hình hội tụ ổn định 100% trên mọi seed.


In [ ]:
import os, time, gc
import torch.nn.functional as F
from datetime import datetime, timezone
from transformers import get_linear_schedule_with_warmup

# Trọng số lớp cân bằng: Tăng ưu tiên cho lớp Refuted (1.20) để triệt tiêu lỗi bỏ sót mâu thuẫn vi mô
class_weights = torch.tensor([1.0, 1.20, 1.0], device=DEVICE)
loss_fn_verdict = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
loss_fn_conflict = nn.CrossEntropyLoss()

DEV_V_TRUES = np.array([s.label_verdict for s in dev_samples])
DEV_C_TRUES = np.array([s.label_conflict for s in dev_samples])


def predict_probs(model, loader):
    # Softmax verdict + conflict theo đúng thứ tự của loader (không shuffle).
    model.eval()
    v_probs, c_probs = [], []
    with torch.no_grad():
        for batch in loader:
            logits_v, logits_c, _ = model(**model_inputs(batch))
            v_probs.append(torch.softmax(logits_v.float(), dim=-1).cpu().numpy())
            c_probs.append(torch.softmax(logits_c.float(), dim=-1).cpu().numpy())
    return np.concatenate(v_probs), np.concatenate(c_probs)


def refuted_conflict_acc(v_trues, c_trues, c_preds):
    idx = np.where(v_trues == 1)[0]
    return accuracy_score(c_trues[idx], c_preds[idx]) if len(idx) else 0.0


def train_one_seed(seed: int, ckpt_path: str):
    set_seed(seed)  # seed quyết định cả khởi tạo head lẫn thứ tự shuffle của train_loader
    model = build_model()

    # 1. PHÂN TÁCH NHÓM THAM SỐ (DIFFERENTIAL LEARNING RATES)
    backbone_params = list(model.backbone.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE, 'weight_decay': 1e-2},
        {'params': head_params, 'lr': LR_HEADS, 'weight_decay': 1e-3}
    ], eps=1e-8)

    effective_steps_per_epoch = len(train_loader) // GRAD_ACCUM
    total_steps = effective_steps_per_epoch * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps
    )
    scaler = torch.cuda.amp.GradScaler()

    best_f1, best_epoch = -1.0, 0
    print(f'\n========== [seed {seed}] Bắt đầu huấn luyện CG-SAP-FC Large ==========')
    print(f' - Backbone LR : {LR_BACKBONE} | Heads LR: {LR_HEADS} | Head Warmup: {HEAD_WARMUP_EPOCHS} Epoch')
    print(f' - Tổng Epochs: {NUM_EPOCHS} | Effective Batch: {BATCH_SIZE*GRAD_ACCUM} | '
          f'Label smoothing: {LABEL_SMOOTHING} | R-Drop alpha: {R_DROP_ALPHA} | Dropout: {DROPOUT}')

    for epoch in range(NUM_EPOCHS):
        # 2. CƠ CHẾ HEAD WARMUP (KHÓA BACKBONE Ở EPOCH ĐẦU)
        backbone_trainable = epoch >= HEAD_WARMUP_EPOCHS
        for p in model.backbone.parameters():
            p.requires_grad = backbone_trainable

        model.train()
        total_loss_verdict = 0.0
        total_loss_consist = 0.0
        total_loss_kl = 0.0
        t0 = time.time()

        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            inputs = model_inputs(batch)
            labels_v = batch['label_verdict'].to(DEVICE)
            labels_c = batch['label_conflict'].to(DEVICE)

            with torch.cuda.amp.autocast():
                if R_DROP_ALPHA > 0:
                    # R-Drop: cùng batch đi qua model 2 lần (dropout khác nhau) trong 1 lượt forward nhân đôi,
                    # rồi phạt KL đối xứng giữa 2 phân phối verdict.
                    doubled = {k: torch.cat([v, v], dim=0) for k, v in inputs.items()}
                    labels_v = torch.cat([labels_v, labels_v], dim=0)
                    labels_c = torch.cat([labels_c, labels_c], dim=0)
                    logits_v, logits_c, gate = model(**doubled)
                    logp_a, logp_b = F.log_softmax(logits_v.float(), dim=-1).chunk(2, dim=0)
                    l_kl = 0.5 * (F.kl_div(logp_a, logp_b, reduction='batchmean', log_target=True)
                                  + F.kl_div(logp_b, logp_a, reduction='batchmean', log_target=True))
                else:
                    logits_v, logits_c, gate = model(**inputs)
                    l_kl = torch.zeros((), device=DEVICE)
                l_v = loss_fn_verdict(logits_v, labels_v)
                l_c = loss_fn_conflict(logits_c, labels_c)

                # Loss nhất quán logic: xác suất conflict cao thì phạt xác suất Supported
                prob_v = torch.softmax(logits_v, dim=-1)
                prob_c = torch.softmax(logits_c, dim=-1)
                prob_conflict_active = 1.0 - prob_c[:, 0]
                l_consist = torch.mean(prob_v[:, 0] * prob_conflict_active)

                loss = (l_v + LAMBDA_CONFLICT * l_c + LAMBDA_CONSISTENCY * l_consist
                        + R_DROP_ALPHA * l_kl) / GRAD_ACCUM

            scaler.scale(loss).backward()
            total_loss_verdict += l_v.item()
            total_loss_consist += l_consist.item()
            total_loss_kl += l_kl.item()

            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad()

        # --- Đánh giá trên Dev Set sau mỗi Epoch ---
        dev_v_probs, dev_c_probs = predict_probs(model, dev_loader)
        dev_v_preds = dev_v_probs.argmax(axis=1)
        dev_macro_f1 = f1_score(DEV_V_TRUES, dev_v_preds, average='macro')
        dev_acc = accuracy_score(DEV_V_TRUES, dev_v_preds)
        c_acc = refuted_conflict_acc(DEV_V_TRUES, DEV_C_TRUES, dev_c_probs.argmax(axis=1))

        phase_str = 'FULL' if backbone_trainable else 'WARMUP'
        print(f'[seed {seed}] Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} [{time.time()-t0:.0f}s] ({phase_str}) - '
              f'Loss(V): {total_loss_verdict/len(train_loader):.3f} | '
              f'Loss(Consist): {total_loss_consist/len(train_loader):.3f} | '
              f'Loss(KL): {total_loss_kl/len(train_loader):.3f} | '
              f'Dev Macro-F1: {dev_macro_f1*100:.2f}% | Dev Acc: {dev_acc*100:.2f}% | '
              f'Conflict-Head Acc: {c_acc*100:.1f}%')

        if dev_macro_f1 > best_f1:
            best_f1, best_epoch = dev_macro_f1, epoch + 1
            torch.save(model.state_dict(), ckpt_path)
            print(f'  --> [BEST] Saved {ckpt_path} (Dev F1: {dev_macro_f1*100:.2f}%)')

    return model, best_f1, best_epoch


def seed_probs_path(seed: int) -> str:
    return os.path.join(OUT_DIR, f'ensemble_probs_seed{seed}.npz')


for seed in ENSEMBLE_SEEDS:
    probs_path = seed_probs_path(seed)
    if os.path.exists(probs_path):
        print(f'[seed {seed}] Đã có xác suất dự đoán tại {probs_path} — bỏ qua.')
        continue

    if seed in EXISTING_CHECKPOINTS:
        ckpt_path = EXISTING_CHECKPOINTS[seed]
        print(f'[seed {seed}] Dùng checkpoint có sẵn: {ckpt_path}')
        set_seed(seed)
        model = build_model()
        best_epoch = -1
    else:
        ckpt_path = os.path.join(OUT_DIR, f'best_multitask_xlmr_large_seed{seed}.pt')
        model, _, best_epoch = train_one_seed(seed, ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    dev_v_probs, dev_c_probs = predict_probs(model, dev_loader)
    test_v_probs, test_c_probs = predict_probs(model, test_loader)
    dev_macro_f1 = f1_score(DEV_V_TRUES, dev_v_probs.argmax(axis=1), average='macro')

    np.savez(
        probs_path,
        dev_verdict=dev_v_probs, dev_conflict=dev_c_probs,
        test_verdict=test_v_probs, test_conflict=test_c_probs,
        best_epoch=best_epoch, dev_macro_f1=dev_macro_f1, checkpoint_path=ckpt_path,
    )
    print(f'[seed {seed}] Best epoch {best_epoch} | Dev Macro-F1 {dev_macro_f1*100:.2f}% | lưu {probs_path}')

    del model
    gc.collect()
    torch.cuda.empty_cache()
    if not KEEP_CHECKPOINTS and seed not in EXISTING_CHECKPOINTS and os.path.exists(ckpt_path):
        os.remove(ckpt_path)
        print(f'[seed {seed}] Đã xóa checkpoint để tiết kiệm Drive.')

print('\nHoàn tất huấn luyện / dự đoán cho tất cả seed:', ENSEMBLE_SEEDS)


## 7. Đánh giá chính thức trên Test Set (ensemble)

In [ ]:
test_labels = ['Supported', 'Refuted', 'NEI']
test_v_trues = np.array([s.label_verdict for s in test_samples])
test_c_trues = np.array([s.label_conflict for s in test_samples])

missing = [s for s in ENSEMBLE_SEEDS if not os.path.exists(seed_probs_path(s))]
if missing:
    raise RuntimeError(f'Chưa có xác suất dự đoán cho seed {missing}. Hãy chạy lại cell huấn luyện.')

per_seed = []
dev_stack, test_stack, test_c_stack = [], [], []
for seed in ENSEMBLE_SEEDS:
    d = np.load(seed_probs_path(seed))
    dev_stack.append(d['dev_verdict'])
    test_stack.append(d['test_verdict'])
    test_c_stack.append(d['test_conflict'])
    seed_test_preds = d['test_verdict'].argmax(axis=1)
    per_seed.append({
        'seed': seed,
        'best_epoch': int(d['best_epoch']),
        'checkpoint_path': str(d['checkpoint_path']),
        'dev_macro_f1': float(d['dev_macro_f1']),
        'test_macro_f1': float(f1_score(test_v_trues, seed_test_preds, average='macro')),
        'test_accuracy': float(accuracy_score(test_v_trues, seed_test_preds)),
    })

# Ensemble = trung bình xác suất softmax của các seed
test_probs = np.mean(test_stack, axis=0)
test_c_probs = np.mean(test_c_stack, axis=0)
dev_probs = np.mean(dev_stack, axis=0)
test_v_preds = test_probs.argmax(axis=1)
test_c_preds = test_c_probs.argmax(axis=1)

best_dev_macro_f1 = f1_score(DEV_V_TRUES, dev_probs.argmax(axis=1), average='macro')
test_macro_f1 = f1_score(test_v_trues, test_v_preds, average='macro')
test_acc = accuracy_score(test_v_trues, test_v_preds)
report = classification_report(test_v_trues, test_v_preds, target_names=test_labels, output_dict=True, digits=4)
cm = confusion_matrix(test_v_trues, test_v_preds)
c_test_acc = refuted_conflict_acc(test_v_trues, test_c_trues, test_c_preds)

ddof = 1 if len(per_seed) > 1 else 0
single_f1 = np.array([r['test_macro_f1'] for r in per_seed])
single_acc = np.array([r['test_accuracy'] for r in per_seed])
single_f1_mean, single_f1_std = single_f1.mean(), single_f1.std(ddof=ddof)
single_acc_mean, single_acc_std = single_acc.mean(), single_acc.std(ddof=ddof)
dev_f1_mean = np.mean([r['dev_macro_f1'] for r in per_seed])

print('\n' + '='*65)
print(f'TỪNG SEED TRÊN {DATASET_NAME.upper()} TEST (n={len(test_samples)})')
print('='*65)
for r in per_seed:
    print(f"  seed {r['seed']:>5} | best epoch {r['best_epoch']:>2} | Dev F1 {r['dev_macro_f1']*100:.2f}% | "
          f"Test F1 {r['test_macro_f1']*100:.2f}% | Test Acc {r['test_accuracy']*100:.2f}%")
print(f'  1 seed (mean ± std): Macro-F1 {single_f1_mean*100:.2f} ± {single_f1_std*100:.2f}% | '
      f'Acc {single_acc_mean*100:.2f} ± {single_acc_std*100:.2f}%')

print('\n' + '='*65)
print(f'KẾT QUẢ ENSEMBLE {len(ENSEMBLE_SEEDS)} SEED - SAP-FC LARGE (n={len(test_samples)})')
print('='*65)
print(f'Dev Macro-F1 : {best_dev_macro_f1*100:.2f}%')
print(f'Macro-F1     : {test_macro_f1*100:.2f}%')
print(f'Accuracy     : {test_acc*100:.2f}%')
for lab in test_labels:
    print(f'  {lab:10s} - P: {report[lab]["precision"]*100:.1f}% | R: {report[lab]["recall"]*100:.1f}% | F1: {report[lab]["f1-score"]*100:.1f}%')
print('='*65)
print(f'Conflict Slot Accuracy (trên tập Refuted Test): {c_test_acc*100:.2f}%')

# ==================== 1. LƯU METRIC VÀO FILE JSON ====================
pct = lambda x: round(float(x) * 100, 2)
metrics_result = {
    'dataset': DATASET_NAME,
    'model_key': MODEL_KEY,
    'num_test_samples': len(test_samples),
    'aggregation': 'mean_softmax',
    'ensemble_seeds': ENSEMBLE_SEEDS,
    'macro_f1': pct(test_macro_f1),
    'accuracy': pct(test_acc),
    'conflict_slot_acc': pct(c_test_acc),
    'best_dev_macro_f1': pct(best_dev_macro_f1),
    'single_seed': {
        'macro_f1_mean': pct(single_f1_mean),
        'macro_f1_std': pct(single_f1_std),
        'accuracy_mean': pct(single_acc_mean),
        'accuracy_std': pct(single_acc_std),
        'dev_macro_f1_mean': pct(dev_f1_mean),
        'runs': [
            {**r, 'dev_macro_f1': pct(r['dev_macro_f1']),
             'test_macro_f1': pct(r['test_macro_f1']), 'test_accuracy': pct(r['test_accuracy'])}
            for r in per_seed
        ],
    },
    'per_class': {
        lab: {
            'precision': pct(report[lab]['precision']),
            'recall': pct(report[lab]['recall']),
            'f1_score': pct(report[lab]['f1-score']),
            'support': int(report[lab]['support']),
        }
        for lab in test_labels
    },
    'confusion_matrix': cm.tolist(),
    'hyperparameters': {
        'max_len': MAX_LEN,
        'batch_size': BATCH_SIZE,
        'grad_accum': GRAD_ACCUM,
        'effective_batch_size': BATCH_SIZE * GRAD_ACCUM,
        'lr_backbone': LR_BACKBONE,
        'lr_heads': LR_HEADS,
        'head_warmup_epochs': HEAD_WARMUP_EPOCHS,
        'epochs': NUM_EPOCHS,
        'dropout': DROPOUT,
        'lambda_conflict': LAMBDA_CONFLICT,
        'lambda_consistency': LAMBDA_CONSISTENCY,
        'run_tag': RUN_TAG,
        'backbone_path': BACKBONE_PATH,
        'nli_intermediate': USE_NLI_INTERMEDIATE,
        'nli_source': NLI_SOURCE if USE_NLI_INTERMEDIATE else None,
        'nli_train_samples': NLI_TRAIN_SAMPLES if USE_NLI_INTERMEDIATE else None,
        'diff_markers': USE_DIFF_MARKERS,
        'pair_pooling': USE_PAIR_POOLING,
        'label_smoothing': LABEL_SMOOTHING,
        'r_drop_alpha': R_DROP_ALPHA,
    },
    'saved_at': datetime.now(timezone.utc).isoformat(),
}

metrics_json_path = os.path.join(OUT_DIR, 'test_metrics_ensemble.json')
with open(metrics_json_path, 'w', encoding='utf-8') as f_m:
    json.dump(metrics_result, f_m, indent=2, ensure_ascii=False)
print(f'\n💾 Đã lưu Test Metrics (ensemble) vào: {metrics_json_path}')

# ==================== 2. LƯU DỰ ĐOÁN TEST VÀO FILE JSON ====================
seed_test_preds = {seed: stack.argmax(axis=1) for seed, stack in zip(ENSEMBLE_SEEDS, test_stack)}
predictions_data = []
for i, s in enumerate(test_samples):
    pred_v_id = int(test_v_preds[i])
    gold_v_id = int(test_v_trues[i])
    pred_c_id = int(test_c_preds[i])
    gold_c_id = int(test_c_trues[i])
    probs = test_probs[i]
    predictions_data.append({
        'id': getattr(s, 'id', str(i)),
        'claim': s.claim,
        'evidence': s.evidence,
        'gold_verdict': ID2LABEL[gold_v_id],
        'pred_verdict': ID2LABEL[pred_v_id],
        'gold_verdict_id': gold_v_id,
        'pred_verdict_id': pred_v_id,
        'is_verdict_correct': bool(gold_v_id == pred_v_id),
        'verdict_probabilities': {
            'Supported': round(float(probs[0]), 4),
            'Refuted': round(float(probs[1]), 4),
            'NEI': round(float(probs[2]), 4),
        },
        'per_seed_pred_verdict': {str(seed): ID2LABEL[int(p[i])] for seed, p in seed_test_preds.items()},
        'gold_conflict': ID2CONFLICT.get(gold_c_id, str(gold_c_id)),
        'pred_conflict': ID2CONFLICT.get(pred_c_id, str(pred_c_id)),
        'gold_conflict_id': gold_c_id,
        'pred_conflict_id': pred_c_id,
        'is_conflict_correct': bool(gold_c_id == pred_c_id),
    })

predictions_json_path = os.path.join(OUT_DIR, 'test_predictions_ensemble.json')
with open(predictions_json_path, 'w', encoding='utf-8') as f_p:
    json.dump(predictions_data, f_p, indent=2, ensure_ascii=False)
print(f'💾 Đã lưu Test Predictions (ensemble) vào: {predictions_json_path} ({len(predictions_data)} mẫu)')


## 9. Bảng kết quả để chèn vào Bảng 5.1 của Paper


In [ ]:
print('=== BẢNG KẾT QUẢ ĐỂ CHÈN VÀO PAPER-SKELETON.MD (BẢNG 5.1) ===\n')
print(f'| Mô hình / Phương pháp | Macro-F1 (Dev) | Macro-F1 (Test) | Accuracy (Test) | Supported F1 | Refuted F1 | NEI F1 | Explainability |')
print(f'|---|---|---|---|---|---|---|---|')
if DATASET_NAME == 'vifactcheck':  # baseline đã công bố chỉ có cho ViFactCheck
    print(f'| XLM-R Base (Paper ViFactCheck) | — | 81.10% | — | — | — | — | ❌ Hộp đen |')
    print(f'| SAP-FC Base (Đã chạy trước) | — | 81.49% | 81.55% | 82.3% | 76.4% | 85.8% | ✅ Slot-Attributed |')
    print(f'| XLM-R Large (B0s Blackbox) | 86.4% | 87.27% | 87.28% | 87.8% | 84.8% | 89.2% | ❌ Hộp đen |')
    print(f'| ViCLSR SCL Large (arXiv 2026) | — | 88.78% | — | — | — | — | ❌ Hộp đen |')
print(f'| SAP-FC Large, 1 seed (mean ± std, {len(ENSEMBLE_SEEDS)} seeds) | {dev_f1_mean*100:.2f}% | {single_f1_mean*100:.2f} ± {single_f1_std*100:.2f}% | {single_acc_mean*100:.2f} ± {single_acc_std*100:.2f}% | — | — | — | ✅ Slot-Attributed Rationale |')
print(f'| **SAP-FC Large Ensemble ({len(ENSEMBLE_SEEDS)} seeds, Đề xuất)** | **{best_dev_macro_f1*100:.2f}%** | **{test_macro_f1*100:.2f}%** | **{test_acc*100:.2f}%** | **{report["Supported"]["f1-score"]*100:.1f}%** | **{report["Refuted"]["f1-score"]*100:.1f}%** | **{report["NEI"]["f1-score"]*100:.1f}%** | ✅ **Slot-Attributed Rationale** |')

print(f'\n=== DANH SÁCH FILE KẾT QUẢ ĐÃ LƯU TẠI: {OUT_DIR} ===')
for f_name in sorted(os.listdir(OUT_DIR)):
    f_path = os.path.join(OUT_DIR, f_name)
    if os.path.isfile(f_path):
        size_kb = os.path.getsize(f_path) / 1024
        size_str = f'{size_kb/1024:.2f} MB' if size_kb > 1024 else f'{size_kb:.1f} KB'
        print(f'  ✅ {f_name:30s} ({size_str})')